# 27장 · 작업 분할 · 같은 초안을 서로 다른 기준으로 검토하기

작업 분할은 한 입력에 필요한 일을 독립적인 호출로 나누는 방법입니다. 같은 문장을 번역하는 일과 글의 분위기를 판단하는 일은 서로의 결과를 기다릴 필요가 없습니다.

## 같은 입력에 다른 지시 주기

같은 글에서 번역과 말투 판정을 따로 받으려면 원문은 그대로 두고 지시만 바꿉니다. 두 번째 호출에도 첫 호출에 넣었던 원문을 그대로 넘깁니다.

`ask(지시, 입력)`은 OpenAI SDK로 요청하고 답을 문자열로 돌려주는 수업용 함수입니다. 수업 커널에서 먼저 아래 셀로 함수를 정의하세요. 새 커널에서 openai를 찾지 못하면 `%pip install openai`를 실행하세요.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

model = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ")

client = OpenAI(timeout=45, max_retries=1)  # 8장에서 준비한 환경 변수 OPENAI_API_KEY를 읽습니다.

def ask(instruction, material):
    response = client.responses.create(
        model=model, instructions=instruction, input=material, store=False
    )
    return response.output_text

In [ ]:
text = "다시 만나서 반가워요."
translation = ask("영어로 번역하세요.", text)
tone = ask("말투를 한 단어로 표현하세요.", text)
print(translation, tone, sep="\n")

두 번째 호출에도 원문 `text`가 들어갑니다. 이렇게 독립적인 작업을 순서대로 먼저 시험한 뒤 다음 장에서 동시에 실행합니다.

## 응용 실습 · 초안의 사실과 표현 따로 검토하기

### 검토할 초안과 원래 문의 준비하기

26장에서 검토 대상으로 분류한 초안을 다시 읽습니다. 같은 문장을 사실과 표현이라는 두 관점에서 읽으면 어떤 의견이 나올까요? 앞 장을 건너뛰면 수업 묶음에 들어 있는 복귀 예시를 읽습니다. 첫 출력에 자료의 출처가 표시됩니다.

In [ ]:
from pathlib import Path
import json
import sys

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "pre-mid/documents/workflow").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/workflow"))
from workbook_files import read_previous, save_record

previous = read_previous(root, 26)
print(previous["route"], previous["draft"])

교재 복귀 예시 · chapter26.json
review 6월 11일 정오까지 원래 신청 게시글에 연장 요청 댓글을 남기세요. 담당자가 승인 댓글을 달아야 반납일이 바뀝니다.


## 사실 검토와 표현 검토의 입력 정하기

두 검토는 같은 초안을 받습니다. 사실 검토에는 대조할 규정을 더하고, 표현 검토에는 초안만 줍니다. 사실 검토의 출력이 없어도 표현 검토의 입력을 만들 수 있는지 확인하세요. 두 호출 사이에 이런 의존성이 없으면 나중에 동시에 실행할 수 있습니다.

## 응용에 사용할 자료 열기

실습 묶음의 압축을 풀고 노트북과 `pre-mid` 폴더를 나란히 두세요. 다음 셀은 대여 규정 파일을 읽어 `rules`에 저장합니다. 응용에서도 앞에서 정의한 `ask`를 사용합니다.

In [ ]:
from pathlib import Path

rules = (next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pre-mid/documents/sources/loan-rules.md").is_file()) / "pre-mid/documents/sources/loan-rules.md").read_text(encoding="utf-8")

In [ ]:
draft = previous["draft"]
revision = "회원안내-초안1"  # 검토 대상 문장을 고치면 다른 이름으로 보관합니다.
print(draft)

## ask에 사실 검토 지시와 근거 문서 전달하기

사실 검토에는 규정 원문이 필요합니다. 근거 조항 이름 하나만 받으면 그 조항이 실제로 그 말을 하는지 알기 어렵습니다. 초안의 문구와 원문의 조건을 짝으로 요구하세요.

여기서는 문의 한 건을 서로 다른 두 작업으로 분할합니다. 두 검토의 공통 입력은 같은 초안 `draft`입니다. 사실 검토의 `ask`에는 규정까지 붙인 `fact_input`을, 표현 검토의 `ask`에는 초안만 줍니다. 두 검토에 필요한 자료는 처음부터 준비돼 있습니다. 어느 쪽을 먼저 호출해도 상대방의 답을 기다릴 필요가 없습니다.

In [ ]:
fact_instruction = '규정과 어긋난 초안 문구를 인용하세요. 각 문구에 근거 조항과 원문의 정확한 조건을 붙이세요. 자료로 확인할 수 없는 것은 확인 필요로 적으세요.'
style_instruction = '첫 참석자가 안내만 읽고 할 일을 알 수 있는지 검토하세요. 누가, 언제, 어디서 무엇을 할지 모호한 표현 하나와 고치는 방법을 쓰세요. 규정상 사실의 참거짓은 판정하지 마세요.'

In [ ]:
fact_input = rules + "\n검토할 초안\n" + draft
fact_review = ask(fact_instruction, fact_input)
print(fact_review)

기본 수령·반납 문의라면 R2의 직접 수령과 R3의 반납 기한을 대조하세요. 본인의 초안이 다르면 그 문의에 해당하는 조항을 읽으세요. “학생증”만 언급했다고 R2에 맞는 것은 아닙니다. 누가 학생증을 제시하는지가 바뀌면 다른 행동을 허용합니다.

> **R2**　승인받은 회원은 대여일 오후 3시부터 5시까지 가온관 1층 안내 데스크에서 기기를 받습니다. 학생증을 보여 주세요. 친구가 대신 받을 수 없습니다.

> **R3**　수령 다음 날 오후 5시까지 가온관 1층 안내 데스크로 반납해 주세요. 기기와 함께 빌린 케이블도 가져오세요.

## 두 번째 ask에도 원래 draft 전달하기

표현 검토에는 같은 초안을 그대로 줍니다. 앞 검토의 수정안을 새 초안처럼 보내면 두 결과가 다른 버전의 글을 가리켜 비교할 수 없습니다. 첫 단계의 결과를 어디에도 넘기지 않은 코드에 주목하세요.

In [ ]:
style_review = ask(style_instruction, draft)
print(style_review)

표현 검토자가 사실 검토처럼 행동하거나 규정에 없는 장소를 제안했다면 그 부분은 채택하지 마세요. 어느 검토에서 나온 말인지 기록하면 담당자는 규정 원문과 독자 편의라는 서로 다른 기준을 되짚을 수 있습니다.

In [ ]:
# 같은 초안 식별자 곁에 두 결과를 둬서 다른 글의 검토가 섞이지 않게 합니다.
reviews = {"초안": revision, "사실": fact_review, "표현": style_review}
print(reviews["초안"], "에서 검토한 원문:", draft)

## 검토가 엇갈리면 무엇을 채택할까

`fact_review`와 `style_review`는 각 호출이 낸 의견입니다. 사실을 고칠 때는 초안과 규정을, 표현을 고칠 때는 같은 초안과 표현 의견을 대조합니다. 의견이 충돌하는 문구 하나를 골라 어떤 원문이 결정을 뒷받침하는지 기록하세요.

표현 검토자가 “기기”를 “빌린 물건”으로 간결하게 바꾸자고 했다면 R3을 다시 읽으세요. 케이블을 명시해야 학생이 실제 반납물에 넣을 가능성이 큽니다. 규정 조건을 살리고 읽기 쉽게 쓰는 한 문장을 직접 만드세요. 두 검토의 의견을 모두 이어 붙여도 완성된 안내가 되지는 않습니다.

## 검토 결과를 실제 게시 문장으로 조합하기

**집필자가 쓴 두 검토의 예시**　사실 담당자는 “학생증을 친구에게 맡기면”을 R2와, “이틀 뒤까지”를 R3과 대조합니다. 표현 담당자는 “가져오면”의 주체와 장소가 빠졌다고 지적합니다. 이를 근거로 “승인받은 회원이 학생증을 가지고 직접 수령해 주세요. 다음 날 오후 5시까지 기기와 케이블을 가온관 1층 안내 데스크에 돌려주세요.”라고 고칠 수 있습니다. 표현 담당자가 규정 밖에서 장소를 지어낸 경우 그 문구는 채택하지 않습니다.

## 검토 의견을 채택할 권한 정하기

R3을 읽은 사실 검토자가 “부속품”을 “케이블”로 적으라고 했습니다. 표현 검토자는 짧게 읽히도록 “모두”로 바꾸자고 합니다. 둘 다 추천했다는 사실만으로 순서를 결정할 수 없습니다. 규정에 필수 반납물이 구체적으로 적혀 있다면 그 물건을 표시하고, 그 조건을 유지하는 범위에서 문장을 짧게 고치세요. 짝에게 둘 중 하나를 지우면 회원이 무엇을 놓칠지 묻습니다. 회신 문장을 고칠 때 검토별 의견과 최종 채택 이유를 구분해 기록하세요.

준비물은 실습 묶음의 `pre-mid/documents/sources/loan-rules.md`와 `pre-mid/documents/questions/loan-inquiries.md`입니다. 짝과 검토할 기록지는 `pre-mid/documents/workflow/workshop.md`에 있습니다.

## 실습 · 친절하지만 틀린 안내와 정확하지만 불친절한 안내

개인 실습 75분. 새 초안 “승인 댓글을 기다려 주셔서 감사합니다! 반납은 편하실 때 해 주세요.”를 두 기준으로 검토합니다. 이어 “수령 익일 17시까지 당해 대여물 및 부속품을 수령처에 반환”을 시험합니다. 각 문구에 대해 채택할 지적과 보류할 지적을 한 개씩 적고, 회원에게 보낼 두 문장으로 다시 씁니다. 두 검토에 실제로 전달된 자료도 제출하세요.

In [ ]:
new_draft = "승인 댓글을 기다려 주셔서 감사합니다! 반납은 편하실 때 해 주세요."
# 같은 new_draft를 두 검토에 넣으세요.
# 사실 검토에는 rules와 초안을 함께, 표현 검토에는 초안만 넣으세요. 두 답에서 각각 어떤 입력을 근거로 삼았는지 찾으세요.

<details class="notebook-answer"><summary>모범 풀이와 대조 기준</summary>
<pre><code>for new_draft in [&quot;승인 댓글을 기다려 주셔서 감사합니다! 반납은 편하실 때 해 주세요.&quot;,
                  &quot;수령 익일 17시까지 당해 대여물 및 부속품을 수령처에 반환&quot;]:
    fact_material = rules + &quot;\n&quot; + new_draft
    print(&quot;사실 검토 입력&quot;, fact_material)
    print(ask(fact_instruction, fact_material))
    print(&quot;표현 검토 입력&quot;, new_draft)
    print(ask(style_instruction, new_draft))</code></pre>
<p>첫 초안의 “편하실 때”는 R3 기한과 맞지 않습니다. 둘째 문장의 익일·당해·수령처는 첫 이용자에게 어렵습니다. 사람이 쓴 수정 예시는 “기기를 받은 다음 날 오후 5시까지 가온관 1층 안내 데스크에 오세요. 기기와 빌린 케이블을 함께 돌려주세요.”입니다.</p>
</details>

## 사실 검토와 표현 검토의 지시 보관하기

두 검토를 같은 초안에 실행한 뒤 사용한 지시와 결과를 남깁니다. 28장에서는 모델에 보낼 초안이나 지시를 다시 쓰지 않아도 됩니다.

In [ ]:
record = {"question": previous["question"], "draft": draft,
          "instructions": {"사실": fact_instruction, "표현": style_instruction},
          "reviews": {"사실": fact_review, "표현": style_review}}
save_record(root, 27, record)

같은 문장을 두 번 검토했어도 평가 기준은 다릅니다. 각 역할이 읽은 지시와 그 역할의 답을 연결해 보세요. 실시간 모델을 실행하지 않았으면 아래 셀만 골라 실행해 교재 예시의 자료 구조를 읽을 수 있습니다. 첫 줄이 파일의 출처입니다.

In [ ]:
from pathlib import Path
import json
import sys

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "pre-mid/documents/workflow").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/workflow"))
from workbook_files import read_previous, save_record

saved = read_previous(root, 27)
for role in saved["instructions"]:
    print(role, saved["reviews"][role])

교재 복귀 예시 · chapter27.json
사실 담당자 승인 뒤 연장됩니다(R4).
표현 댓글을 남길 게시글을 지정했습니다.


## 다음 작업에 가져갈 기록

어느 검토도 다른 검토의 결과를 입력으로 쓰지 않았습니다. 다음 장에서는 이 독립성을 이용해 두 요청을 함께 시작하고, 하나만 실패했을 때 무엇을 남길지 정합니다.

[OpenAI Python SDK 사용 예](https://github.com/openai/openai-python)